In [1]:
import json
from datetime import date
from pathlib import Path

import requests

URL = "https://sedeaplicaciones.minetur.gob.es/ServiciosRESTCarburantes/PreciosCarburantes/EstacionesTerrestres/"

resp = requests.get(URL, headers={"Accept": "application/json"}, timeout=60)
resp.raise_for_status()
datos = resp.json()

ruta = Path("../data/raw") / f"precios_{date.today():%Y-%m-%d}.json"
ruta.parent.mkdir(parents=True, exist_ok=True)
ruta.write_text(json.dumps(datos, ensure_ascii=False), encoding="utf-8")

print(ruta, f"{ruta.stat().st_size / 1e6:.1f} MB")

..\data\raw\precios_2026-10-19.json 13.1 MB


In [ ]:
import requests

URL = "https://sedeaplicaciones.minetur.gob.es/ServiciosRESTCarburantes/PreciosCarburantes/EstacionesTerrestres/"

headers = {
    "Accept": "application/json",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0 Safari/537.36",
}

resp = requests.get(URL, headers=headers, timeout=60)
print(resp.status_code, len(resp.content) / 1e6, "MB")"MB")

In [ ]:
import requests

URL = "https://sedeaplicaciones.minetur.gob.es/ServiciosRESTCarburantes/PreciosCarburantes/EstacionesTerrestres/"

headers = {
    "Accept": "application/json",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0 Safari/537.36",
}

resp = requests.get(URL, headers=headers, timeout=60)
print(resp.status_code, len(resp.content) / 1e6, "MB")

In [ ]:
import ssl
import requests
from requests.adapters import HTTPAdapter

URL = "https://sedeaplicaciones.minetur.gob.es/ServiciosRESTCarburantes/PreciosCarburantes/EstacionesTerrestres/"
HEADERS = {"Accept": "application/json"}

class CtxAdapter(HTTPAdapter):
    """Adapter que usa un contexto TLS concreto para las conexiones."""
    def __init__(self, ctx, **kwargs):
        self._ctx = ctx
        super().__init__(**kwargs)

    def init_poolmanager(self, *args, **kwargs):
        kwargs["ssl_context"] = self._ctx
        return super().init_poolmanager(*args, **kwargs)

def ctx_por_defecto():
    return ssl.create_default_context()

def ctx_tls12():
    ctx = ssl.create_default_context()
    ctx.maximum_version = ssl.TLSVersion.TLSv1_2
    return ctx

def ctx_tls12_cifrados_amplios():
    ctx = ctx_tls12()
    ctx.set_ciphers("DEFAULT:@SECLEVEL=1")
    return ctx

def ctx_tls12_cifrados_rsa():
    ctx = ctx_tls12()
    ctx.set_ciphers("AES256-GCM-SHA384:AES128-GCM-SHA256:AES256-SHA256:AES128-SHA256:AES256-SHA:AES128-SHA")
    return ctx

print(ssl.OPENSSL_VERSION)

pruebas = [
    ("por defecto", ctx_por_defecto),
    ("TLS 1.2", ctx_tls12),
    ("TLS 1.2 + amplios", ctx_tls12_cifrados_amplios),
    ("TLS 1.2 + RSA", ctx_tls12_cifrados_rsa),
]

for nombre, crear_ctx in pruebas:
    sesion = requests.Session()
    sesion.mount("https://", CtxAdapter(crear_ctx()))
    try:
        r = sesion.get(URL, headers=HEADERS, timeout=30, stream=True)
        print(f"{nombre:20} OK    {r.status_code}")
        r.close()
    except Exception as e:
        print(f"{nombre:20} FALLA {type(e).__name__}")

In [ ]:
import json
import ssl
from datetime import date
from pathlib import Path

import requests
from requests.adapters import HTTPAdapter

URL = "https://sedeaplicaciones.minetur.gob.es/ServiciosRESTCarburantes/PreciosCarburantes/EstacionesTerrestres/"
HEADERS = {"Accept": "application/json"}

# El servidor del Ministerio solo acepta intercambio de claves RSA en TLS 1.2.
# Se ofrecen primero cifrados modernos (ECDHE) por si el servidor se actualiza.
CIFRADOS = "ECDHE+AESGCM:ECDHE+CHACHA20:AES256-GCM-SHA384:AES128-GCM-SHA256"

class MineturAdapter(HTTPAdapter):
    def init_poolmanager(self, *args, **kwargs):
        ctx = ssl.create_default_context()
        ctx.maximum_version = ssl.TLSVersion.TLSv1_2
        ctx.set_ciphers(CIFRADOS)
        kwargs["ssl_context"] = ctx
        return super().init_poolmanager(*args, **kwargs)

sesion = requests.Session()
sesion.mount("https://sedeaplicaciones.minetur.gob.es/", MineturAdapter())

resp = sesion.get(URL, headers=HEADERS, timeout=60)
resp.raise_for_status()
datos = resp.json()

ruta = Path("../data/raw") / f"precios_{date.today():%Y-%m-%d}.json"
ruta.parent.mkdir(parents=True, exist_ok=True)
ruta.write_text(json.dumps(datos, ensure_ascii=False), encoding="utf-8")

print(ruta, f"{ruta.stat().st_size / 1e6:.1f} MB")

In [ ]:
print(json.dumps(datos["ListaEESSPrecio"][0], indent=2, ensure_ascii=False))

In [ ]:
import pandas as pd

df = pd.DataFrame(datos["ListaEESSPrecio"])
df.head()

In [ ]:
import pandas as pd

df = pd.DataFrame(datos.keys())
df.head()

In [ ]:
import pandas as pd

df = pd.DataFrame(datos.keys())

In [ ]:
import pandas as pd

df = pd.DataFrame(datos["ListaEESSPrecio"])
df.shape

In [ ]:
import pandas as pd

df = pd.DataFrame(datos["ListaEESSPrecio"])
df.dtypes

In [ ]:
import pandas as pd

df = pd.DataFrame(datos["ListaEESSPrecio"])
(df == "").mean()

In [ ]:
import pandas as pd

df = pd.DataFrame(datos["ListaEESSPrecio"])
(df == "0.000000").mean()

In [ ]:
import pandas as pd

df = pd.DataFrame(datos["ListaEESSPrecio"])
df["IDEESS"].duplicated().sum()

In [ ]:
import pandas as pd

df = pd.DataFrame(datos["ListaEESSPrecio"])
df["Margen"].value_counts()

In [ ]:
import pandas as pd

df = pd.DataFrame(datos["ListaEESSPrecio"])
df["Remision"].value_counts()

In [ ]:
import pandas as pd

df = pd.DataFrame(datos["ListaEESSPrecio"])
df["Remisión"].value_counts()

In [ ]:
import pandas as pd

df = pd.DataFrame(datos["ListaEESSPrecio"])
df["Tipo Venta"].value_counts()

In [ ]:
import pandas as pd

df = pd.DataFrame(datos["ListaEESSPrecio"])
df.isna().mean().sort_values()

In [ ]:
print(json.dumps(datos["ListaEESSPrecio"][0], indent=2, ensure_ascii=False))

In [ ]:
valor = df.loc[0, "Precio Hidrogeno"]
print(repr(valor), type(valor), valor == "")

In [ ]:
valor = df.loc[0, "Precio Hidrogeno"]

In [ ]:
print((df["Precio Hidrogeno"] == "").sum())
print((df == "").mean().sort_values(ascending=False).head(10))

In [ ]:
cols_precio = [c for c in df.columns if c.startswith("Precio")]

venden = (df[cols_precio] != "").mean().sort_values(ascending=False)
(venden * 100).round(1)

In [ ]:
import pandas as pd

df = pd.DataFrame(datos["ListaEESSPrecio"])
df.replace("", pd.NA)

In [ ]:
import pandas as pd

df = pd.DataFrame(datos["ListaEESSPrecio"])
gasoleo_a=df.replace("", pd.NA)
gasoleo_a.describe()

In [ ]:
import pandas as pd

df = pd.DataFrame(datos["ListaEESSPrecio"])
gasoleo_a=df["Precio Gasoleo A"].replace("", pd.NA)
gasoleo_a=gasoleo_a.str.replace(",", ".")
gasoleo_a=gasoleo_a.astype(float)
gasoleo_a.describe()

In [ ]:
precios = df.assign(precio=gasoleo_a)

(precios.groupby("Provincia")["precio"]
        .agg(["mean", "count"])
        .sort_values("mean")
        .head(10))

In [ ]:
precios["Rótulo"].value_counts().head(30)

In [ ]:
print(precios["Rótulo"].nunique())

colas = precios["Rótulo"].value_counts()
colas[colas <= 2].sample(30, random_state=1)

In [ ]:
import pandas as pd

df = pd.DataFrame(datos["ListaEESSPrecio"])
latitud=df["Latitud"].replace("", pd.NA)
latitud=latitud.str.replace(",", ".")
latitud=latitud.astype(float)
latitud.describe()

In [ ]:
import pandas as pd

def a_numero(serie):
    """Convierte texto con coma decimal en número. Las cadenas vacías pasan a nulo."""
    return (serie.replace("", pd.NA)
                 .str.replace(",", ".")
                 .astype(float))

lat = a_numero(df["Latitud"])
lon = a_numero(df["Longitud (WGS84)"])

print("LATITUD")
print(lat.describe(), "\n")
print("LONGITUD")
print(lon.describe(), "\n")

# Rangos aproximados de España
en_peninsula = lat.between(36, 44) & lon.between(-9.5, 4.5)
en_canarias = lat.between(27.5, 29.5) & lon.between(-18.5, -13)
fuera = ~(en_peninsula | en_canarias)

print(f"Península y Baleares: {en_peninsula.sum()}")
print(f"Canarias:             {en_canarias.sum()}")
print(f"Fuera de rango:       {fuera.sum()}")

df.loc[fuera, ["IDEESS", "Rótulo", "Localidad", "Provincia", "Latitud", "Longitud (WGS84)"]]

In [ ]:
df.loc[fuera, "Provincia"].value_counts()

In [ ]:
en_ceuta_melilla = lat.between(35, 36) & lon.between(-5.5, -2.8)
errores = fuera & ~en_ceuta_melilla

# Probamos a intercambiar latitud y longitud
lat_girada, lon_girada = lon, lat
arreglada = lat_girada.between(36, 44) & lon_girada.between(-9.5, 4.5)

pd.DataFrame({
    "Rótulo": df["Rótulo"],
    "Localidad": df["Localidad"],
    "Provincia": df["Provincia"],
    "lat": lat,
    "lon": lon,
    "girada_ok": arreglada,
})[errores]